In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor 

from sklearn.ensemble import   GradientBoostingRegressor, StackingRegressor


from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.preprocessing import OneHotEncoder , StandardScaler, LabelEncoder
from sklearn.compose import make_column_transformer , make_column_selector

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score,r2_score  
from tqdm import tqdm

from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")
import os 
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets\Cases\Medical_Cost_Personal')

In [11]:
insurance = pd.read_csv('insurance.csv')
insurance

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [12]:
X,y = insurance.drop('charges', axis = 1) , insurance['charges']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26 )


In [28]:

ohc = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')

ct = make_column_transformer(
                            (ohc,           make_column_selector(dtype_include = object)),
                            ('passthrough', make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False).set_output(transform = 'pandas')
 
scaler = StandardScaler()


In [29]:
lr = LinearRegression()
rdg = Ridge()
las = Lasso()
kn = KNeighborsRegressor()

dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)


pipe_knn = Pipeline([("SCL", scaler),('knn',kn)])


In [33]:
stack = StackingRegressor(estimators = [
                                        ('LR',lr),
                                        ('KNN', pipe_knn),
                                        ('RIDGE',rdg),
                                        ('LAS', las),
                                        ('TREE', dtr)
] , final_estimator = gbm)

pipe = Pipeline([('OHE',ct), ('STACK', stack)])


In [34]:
stack = StackingRegressor(estimators = [
                                        ('LR',lr),
                                        ('KNN', pipe_knn),
                                        ('RIDGE',rdg),
                                        ('LAS', las),
                                        ('TREE', dtr)
] , final_estimator = gbm, passthrough = True)

pipe = Pipeline([('OHE',ct), ('STACK', stack)])


In [31]:
pipe.fit(X_train, y_train)
y_pred  = pipe.predict(X_test)
r2_score(y_test, y_pred)

0.8596764057963276

In [35]:
# XGBoost

In [37]:
gbm = XGBRegressor(random_state = 26)

stack = StackingRegressor(estimators = [
                                        ('LR',lr),
                                        ('KNN', pipe_knn),
                                        ('RIDGE',rdg),
                                        ('LAS', las),
                                        ('TREE', dtr)
] , final_estimator = gbm, passthrough = True)

pipe = Pipeline([('OHE',ct), ('STACK', stack)])

In [38]:
pipe.fit(X_train, y_train)
y_pred  = pipe.predict(X_test)
r2_score(y_test, y_pred)

0.8360982053278274